# AgriSathi — Member 1: Image-Based Crop Disease Detection

**Hyperlocal Crop Disease Detection for Smallholder Farmers Without Reliable Internet**

This notebook runs the complete Member 1 pipeline:

`Image → Quality Check → Preprocessing → Disease Classification → Confidence + Top-3 → TFLite → INT8 Quantization → Offline Deployment`

Run cells **top to bottom** in Google Colab. Your dataset stays in Google Drive — nothing here uploads or embeds the actual images.


## 1. Environment Setup

Make this project's `config/` and `src/` packages importable, then print environment info and set the reproducibility seed.

In [ ]:
import os, sys

# If you placed this project inside Google Drive, update PROJECT_ROOT
# accordingly (e.g. "/content/gdrive/MyDrive/AgriSathi_Member1").
# If you uploaded config/ and src/ directly into /content/, leave as is.
PROJECT_ROOT = "/content/AgriSathi_Member1"

if os.path.isdir(PROJECT_ROOT):
    sys.path.insert(0, PROJECT_ROOT)
else:
    # Fallback: assume the notebook itself lives at the project root
    sys.path.insert(0, os.getcwd())

from config import config
from src import utils

utils.print_environment_info()
utils.set_global_seed(config.SEED)


## 2. Google Drive Mount

Mounts Google Drive (handles the case where it's already mounted) and verifies `RAW_DIR` exists.

In [ ]:
utils.mount_google_drive(config.DRIVE_MOUNT)
config.ensure_project_dirs()
utils.verify_path_exists(config.RAW_DIR, description="RAW_DIR (raw dataset)", raise_on_missing=False)


## 3. Configuration

All paths and hyperparameters come from `config/config.py` — the single master configuration used by every cell below.
Edit that file (not this notebook) if your Drive layout or hyperparameters differ.

In [ ]:
print("PROJECT_DIR      :", config.PROJECT_DIR)
print("RAW_DIR          :", config.RAW_DIR)
print("SELECTED_DIR     :", config.SELECTED_DIR)
print("TRAIN_DIR        :", config.TRAIN_DIR)
print("VAL_DIR          :", config.VAL_DIR)
print("TEST_DIR         :", config.TEST_DIR)
print("FIELD_TEST_DIR   :", config.FIELD_TEST_DIR)
print("MODEL_DIR        :", config.MODEL_DIR)
print("RESULT_DIR       :", config.RESULT_DIR)
print("IMAGE_SIZE       :", config.IMAGE_SIZE)
print("BATCH_SIZE       :", config.BATCH_SIZE)
print("SEED             :", config.SEED)
print("SELECTED_CLASSES :", config.SELECTED_CLASSES)


## 4. Dataset Inspection

Recursively inspects `RAW_DIR`: actual class names, actual image counts, empty folders, unsupported files. Nothing here is hardcoded — it reflects what is actually in your Drive.

In [ ]:
from src import dataset

summary = dataset.inspect_dataset(config.RAW_DIR, config.SUPPORTED_EXTENSIONS)
dataset.print_dataset_summary(config.RAW_DIR, summary)


In [ ]:
# Verify the six selected classes actually exist before proceeding.
# Stops with a clear error (no silent substitution) if any are missing.
dataset.verify_selected_classes(config.RAW_DIR, config.SELECTED_CLASSES)


## 5. Dataset Quality Check

Checks corrupted/unreadable images, image modes/dimensions, extremely small images, duplicate filenames, exact duplicates, and (if `imagehash` is installed) near-duplicates — for the six selected classes only. Nothing is deleted.

In [ ]:
quality_report = dataset.run_quality_report(
    config.RAW_DIR,
    config.SELECTED_CLASSES,
    config.SUPPORTED_EXTENSIONS,
    min_dimension=config.MIN_IMAGE_DIMENSION,
    check_duplicates=True,
)
dataset.print_quality_report(quality_report)


## 6. EDA

Class distribution table + bar chart, image dimension statistics, and a sample-image grid. Charts save to `Results/Graphs/`.

In [ ]:
from src import eda

class_dir_map = {cls: os.path.join(config.RAW_DIR, cls) for cls in config.SELECTED_CLASSES}

rows, total = eda.class_distribution_table(summary["class_counts"])
print(f"{'Class':<45}{'Images'}")
print("-" * 55)
for cls, count in rows:
    if cls in config.SELECTED_CLASSES:
        print(f"{cls:<45}{count}")
print("-" * 55)
print(f"{'TOTAL (selected classes)':<45}{sum(summary['class_counts'][c] for c in config.SELECTED_CLASSES if c in summary['class_counts'])}")


In [ ]:
selected_counts = {c: summary["class_counts"][c] for c in config.SELECTED_CLASSES if c in summary["class_counts"]}
eda.plot_class_distribution(
    selected_counts,
    save_path=os.path.join(config.GRAPH_DIR, "class_distribution.png"),
    title="AgriSathi — Selected Classes Distribution",
)


In [ ]:
dims = eda.collect_image_dimensions(class_dir_map, config.SUPPORTED_EXTENSIONS, max_per_class=300)
dim_stats = eda.image_dimension_stats(dims)
eda.print_dimension_stats(dim_stats)


In [ ]:
eda.plot_sample_images(
    class_dir_map,
    save_path=os.path.join(config.GRAPH_DIR, "sample_images_per_class.png"),
    supported_extensions=config.SUPPORTED_EXTENSIONS,
    samples_per_class=1,
)


## 7. Selected Dataset

Copies only the six selected classes into `Dataset/Selected_Classes/`. Safe to re-run — will not duplicate files. `RAW_DIR` is never modified.

In [ ]:
prep_summary = dataset.prepare_selected_dataset(
    config.RAW_DIR, config.SELECTED_DIR, config.SELECTED_CLASSES, config.SUPPORTED_EXTENSIONS
)
print(f"Copied: {prep_summary['copied']}, Skipped (already existed): {prep_summary['skipped_existing']}")
print(f"Total images: {prep_summary['total_images']} across {prep_summary['num_classes']} classes")
print(prep_summary["class_counts"])


## 8. Train / Validation / Test Split

Reproducible, stratified, class-aware 70/15/15 split — performed **before** any augmentation. Verifies no leakage across splits afterward using exact file hashes.

In [ ]:
split_summary = dataset.stratified_split(
    config.SELECTED_DIR, config.TRAIN_DIR, config.VAL_DIR, config.TEST_DIR,
    config.SELECTED_CLASSES, config.SUPPORTED_EXTENSIONS,
    train_split=config.TRAIN_SPLIT, val_split=config.VAL_SPLIT, test_split=config.TEST_SPLIT,
    seed=config.SEED,
)


In [ ]:
leak_report = dataset.verify_no_leakage(config.TRAIN_DIR, config.VAL_DIR, config.TEST_DIR, config.FIELD_TEST_DIR)
assert leak_report["num_leaks"] == 0, "Data leakage detected — inspect leak_report['leaks'] before continuing."


## 9. TensorFlow Dataset Pipeline

Builds `tf.data.Dataset` pipelines via `image_dataset_from_directory` — batched, prefetched, and never loading the whole dataset into RAM. `class_names` order here is authoritative for every model's output indices.

In [ ]:
from src import preprocessing

train_ds, val_ds, test_ds, class_names = preprocessing.build_datasets(
    config.TRAIN_DIR, config.VAL_DIR, config.TEST_DIR,
    config.IMAGE_SIZE, config.BATCH_SIZE, config.SEED,
)
print("Detected classes (output-index order):", class_names)


## 10. Augmentation

Realistic, non-extreme augmentation (flip, rotation, zoom, translation, contrast). Applied only inside the training-time model graph — validation/test data is never augmented.

In [ ]:
augmentation_layer = preprocessing.build_augmentation_layer()
augmentation_layer.summary()


## 11. CNN Baseline

Trains the lightweight from-scratch CNN and saves the best checkpoint + training curves.

In [ ]:
from src import models, training

input_shape = config.IMAGE_SIZE + (config.IMAGE_CHANNELS,)

cnn_model = models.build_cnn_baseline(input_shape, num_classes=len(class_names), augmentation_layer=augmentation_layer)
training.compile_model(cnn_model, learning_rate=config.HEAD_LEARNING_RATE)
cnn_model.summary()


In [ ]:
cnn_checkpoint_path = os.path.join(config.CNN_MODEL_DIR, "best_model.keras")
cnn_callbacks = training.build_callbacks(
    cnn_checkpoint_path, patience=config.EARLY_STOPPING_PATIENCE,
    reduce_lr_patience=config.REDUCE_LR_PATIENCE, reduce_lr_factor=config.REDUCE_LR_FACTOR,
    min_lr=config.MIN_LR,
)
cnn_history = training.train_model(cnn_model, train_ds, val_ds, epochs=config.MAX_EPOCHS_CNN, callbacks=cnn_callbacks)


In [ ]:
training.save_history(cnn_history, os.path.join(config.METRICS_DIR, "cnn_baseline_history.json"))
training.plot_training_history(cnn_history.history, "cnn_baseline", config.GRAPH_DIR)


## 12. MobileNetV2

Two-stage transfer learning: frozen-base head training, then fine-tuning the top layers at a low learning rate.

In [ ]:
mobilenetv2_model, mobilenetv2_base = models.build_mobilenetv2(
    input_shape, num_classes=len(class_names), augmentation_layer=augmentation_layer
)
mobilenetv2_checkpoint_path = os.path.join(config.MOBILENETV2_MODEL_DIR, "best_model.keras")

# Stage 1: frozen base
training.compile_model(mobilenetv2_model, learning_rate=config.HEAD_LEARNING_RATE)
callbacks_s1 = training.build_callbacks(mobilenetv2_checkpoint_path, patience=config.EARLY_STOPPING_PATIENCE,
                                        reduce_lr_patience=config.REDUCE_LR_PATIENCE, reduce_lr_factor=config.REDUCE_LR_FACTOR,
                                        min_lr=config.MIN_LR)
history_v2_s1 = training.train_model(mobilenetv2_model, train_ds, val_ds, epochs=config.MAX_EPOCHS_HEAD, callbacks=callbacks_s1)
training.save_history(history_v2_s1, os.path.join(config.METRICS_DIR, "mobilenetv2_stage1_history.json"))


In [ ]:
# Stage 2: fine-tune top layers
models.unfreeze_top_layers(mobilenetv2_base, config.FINE_TUNE_UNFREEZE_LAYERS)
training.compile_model(mobilenetv2_model, learning_rate=config.FINE_TUNE_LEARNING_RATE)
callbacks_s2 = training.build_callbacks(mobilenetv2_checkpoint_path, patience=config.EARLY_STOPPING_PATIENCE,
                                        reduce_lr_patience=config.REDUCE_LR_PATIENCE, reduce_lr_factor=config.REDUCE_LR_FACTOR,
                                        min_lr=config.MIN_LR)
history_v2_s2 = training.train_model(mobilenetv2_model, train_ds, val_ds, epochs=config.MAX_EPOCHS_FINE_TUNE, callbacks=callbacks_s2)
training.save_history(history_v2_s2, os.path.join(config.METRICS_DIR, "mobilenetv2_stage2_history.json"))
training.plot_training_history(history_v2_s2.history, "mobilenetv2", config.GRAPH_DIR)


## 13. MobileNetV3

Same two-stage strategy using MobileNetV3Small, which handles its own input rescaling internally (`include_preprocessing=True`) — no separate `preprocess_input` call is applied, to avoid double normalization.

In [ ]:
mobilenetv3_model, mobilenetv3_base = models.build_mobilenetv3_small(
    input_shape, num_classes=len(class_names), augmentation_layer=augmentation_layer
)
mobilenetv3_checkpoint_path = os.path.join(config.MOBILENETV3_MODEL_DIR, "best_model.keras")

training.compile_model(mobilenetv3_model, learning_rate=config.HEAD_LEARNING_RATE)
callbacks_v3_s1 = training.build_callbacks(mobilenetv3_checkpoint_path, patience=config.EARLY_STOPPING_PATIENCE,
                                           reduce_lr_patience=config.REDUCE_LR_PATIENCE, reduce_lr_factor=config.REDUCE_LR_FACTOR,
                                           min_lr=config.MIN_LR)
history_v3_s1 = training.train_model(mobilenetv3_model, train_ds, val_ds, epochs=config.MAX_EPOCHS_HEAD, callbacks=callbacks_v3_s1)
training.save_history(history_v3_s1, os.path.join(config.METRICS_DIR, "mobilenetv3_stage1_history.json"))


In [ ]:
models.unfreeze_top_layers(mobilenetv3_base, config.FINE_TUNE_UNFREEZE_LAYERS)
training.compile_model(mobilenetv3_model, learning_rate=config.FINE_TUNE_LEARNING_RATE)
callbacks_v3_s2 = training.build_callbacks(mobilenetv3_checkpoint_path, patience=config.EARLY_STOPPING_PATIENCE,
                                           reduce_lr_patience=config.REDUCE_LR_PATIENCE, reduce_lr_factor=config.REDUCE_LR_FACTOR,
                                           min_lr=config.MIN_LR)
history_v3_s2 = training.train_model(mobilenetv3_model, train_ds, val_ds, epochs=config.MAX_EPOCHS_FINE_TUNE, callbacks=callbacks_v3_s2)
training.save_history(history_v3_s2, os.path.join(config.METRICS_DIR, "mobilenetv3_stage2_history.json"))
training.plot_training_history(history_v3_s2.history, "mobilenetv3", config.GRAPH_DIR)


## 14. Model Evaluation

Evaluates every trained model on the **same** test set. If a model's checkpoint doesn't exist yet (not trained), it is skipped with `"Not evaluated yet."` rather than fabricating a result.

In [ ]:
import tensorflow as tf
from src import evaluation

MODEL_CHECKPOINTS = {
    "CNN": os.path.join(config.CNN_MODEL_DIR, "best_model.keras"),
    "MobileNetV2": os.path.join(config.MOBILENETV2_MODEL_DIR, "best_model.keras"),
    "MobileNetV3": os.path.join(config.MOBILENETV3_MODEL_DIR, "best_model.keras"),
}

eval_results = {}
comparison_rows = []

for model_name, ckpt_path in MODEL_CHECKPOINTS.items():
    print(f"\n{'='*60}\nEVALUATING: {model_name}\n{'='*60}")
    if not os.path.exists(ckpt_path):
        print(f"Not evaluated yet. (checkpoint not found: {ckpt_path})")
        continue

    m = tf.keras.models.load_model(ckpt_path)
    y_true, y_pred, y_probs = evaluation.get_predictions(m, test_ds)
    metrics = evaluation.compute_metrics(y_true, y_pred, class_names)
    print(metrics["report_text"])
    evaluation.save_classification_report(
        metrics["report_text"], os.path.join(config.METRICS_DIR, f"{model_name}_classification_report.txt")
    )
    eval_results[model_name] = {"model": m, "y_true": y_true, "y_pred": y_pred, "y_probs": y_probs, "metrics": metrics}


## 15. Confusion Matrices

Raw + row-normalized confusion matrices for every model that was actually evaluated above, using real class names.

In [ ]:
for model_name, r in eval_results.items():
    evaluation.plot_confusion_matrix(
        r["y_true"], r["y_pred"], class_names,
        os.path.join(config.CONF_MATRIX_DIR, f"{model_name}_confusion_matrix.png"),
        normalize=False, title=f"{model_name} — Confusion Matrix",
    )
    evaluation.plot_confusion_matrix(
        r["y_true"], r["y_pred"], class_names,
        os.path.join(config.CONF_MATRIX_DIR, f"{model_name}_confusion_matrix_normalized.png"),
        normalize=True, title=f"{model_name} — Normalized Confusion Matrix",
    )


## 16. Model Comparison

Builds `Results/Metrics/model_comparison.csv` from actual measured metrics, parameter counts, model size, and CPU-oriented inference latency. Only models evaluated above are included.

In [ ]:
from src.models import count_parameters

for model_name, r in eval_results.items():
    m = r["model"]
    metrics = r["metrics"]
    ckpt_path = MODEL_CHECKPOINTS[model_name]

    total_params, trainable_params = count_parameters(m)
    model_size_mb = evaluation.get_model_file_size_mb(ckpt_path)
    sample_batch = next(iter(test_ds.take(1)))[0][:1].numpy()
    latency = evaluation.measure_inference_latency(m, sample_batch)

    comparison_rows.append({
        "Model": model_name,
        "Accuracy": round(metrics["accuracy"], 4),
        "Macro_Precision": round(metrics["macro_precision"], 4),
        "Macro_Recall": round(metrics["macro_recall"], 4),
        "Macro_F1": round(metrics["macro_f1"], 4),
        "Weighted_Precision": round(metrics["weighted_precision"], 4),
        "Weighted_Recall": round(metrics["weighted_recall"], 4),
        "Weighted_F1": round(metrics["weighted_f1"], 4),
        "Total_Parameters": total_params,
        "Trainable_Parameters": trainable_params,
        "Model_Size_MB": round(model_size_mb, 2) if model_size_mb else None,
        "CPU_Inference_Time_ms": round(latency["mean_ms"], 2),
    })

if comparison_rows:
    comparison_df = evaluation.build_comparison_table(
        comparison_rows, save_path=os.path.join(config.METRICS_DIR, "model_comparison.csv")
    )
    display(comparison_df)
else:
    print("No models evaluated yet — nothing to compare. Not evaluated yet.")


### Choosing the deployment candidate

Pick the model that best balances **accuracy, macro-F1, model size, and inference latency together** — not simply the highest accuracy. Review `comparison_df` above, then set `DEPLOYMENT_MODEL_NAME` in the TFLite sections below (Section 20 onward).

## 17. Error Analysis

Identifies misclassified test images for a chosen model and reports the most common confusion pairs. Uses actual model predictions only — no invented explanations.

In [ ]:
ERROR_ANALYSIS_MODEL = "MobileNetV2"  # change to whichever model you want to inspect

if ERROR_ANALYSIS_MODEL in eval_results:
    r = eval_results[ERROR_ANALYSIS_MODEL]
    file_paths = test_ds.file_paths if hasattr(test_ds, "file_paths") else []
    error_df = evaluation.error_analysis(
        r["y_true"], r["y_pred"], r["y_probs"], file_paths, class_names,
        save_path=os.path.join(config.ERROR_DIR, "error_analysis.csv"),
    )
else:
    print(f"Not evaluated yet: {ERROR_ANALYSIS_MODEL} has no checkpoint/evaluation results.")


## 18. Field Test Evaluation

Evaluates the field-test set (`Dataset/Field_Test/`) if real field images have been added. If the folder is empty, prints a clear message instead of fabricating results — it will work automatically once you add real field images with the same class-folder structure.

In [ ]:
def has_field_test_images(field_test_dir, selected_classes):
    if not os.path.isdir(field_test_dir):
        return False
    for cls in selected_classes:
        cls_dir = os.path.join(field_test_dir, cls)
        if os.path.isdir(cls_dir) and len(os.listdir(cls_dir)) > 0:
            return True
    return False


if has_field_test_images(config.FIELD_TEST_DIR, config.SELECTED_CLASSES):
    field_ds = tf.keras.utils.image_dataset_from_directory(
        config.FIELD_TEST_DIR, image_size=config.IMAGE_SIZE, batch_size=config.BATCH_SIZE,
        shuffle=False, label_mode="int",
    )
    if field_ds.class_names != class_names:
        print("[WARNING] Field-test class folders don't exactly match the training classes; "
              "align folder names before evaluating.")
    else:
        for model_name, ckpt_path in MODEL_CHECKPOINTS.items():
            if model_name not in eval_results:
                continue
            m = eval_results[model_name]["model"]
            y_true_f, y_pred_f, _ = evaluation.get_predictions(m, field_ds)
            field_metrics = evaluation.compute_metrics(y_true_f, y_pred_f, class_names)
            print(f"\n=== {model_name} — Field Test ===")
            print(field_metrics["report_text"])
else:
    print("No field-test images available yet. "
          "Add real smartphone-captured images to Dataset/Field_Test/<class_name>/ "
          "and re-run this cell — the evaluation pipeline is already in place.")


## 19. Final Model Selection

Set the deployment candidate here based on the comparison table above (Section 16). This name is used by every TFLite cell below.

In [ ]:
DEPLOYMENT_MODEL_NAME = "MobileNetV2"  # <-- change based on Results/Metrics/model_comparison.csv

assert DEPLOYMENT_MODEL_NAME in MODEL_CHECKPOINTS, f"Unknown model name: {DEPLOYMENT_MODEL_NAME}"
deployment_checkpoint_path = MODEL_CHECKPOINTS[DEPLOYMENT_MODEL_NAME]
assert os.path.exists(deployment_checkpoint_path), (
    f"{DEPLOYMENT_MODEL_NAME} has not been trained yet — checkpoint missing at {deployment_checkpoint_path}"
)
deployment_model = tf.keras.models.load_model(deployment_checkpoint_path)
print(f"[OK] Deployment candidate: {DEPLOYMENT_MODEL_NAME}")


## 20. TFLite Float32

Converts the chosen deployment model to a plain float32 TFLite model.

In [ ]:
from src import tflite_utils

os.makedirs(config.FINAL_MODEL_DIR, exist_ok=True)
float32_path = os.path.join(config.FINAL_MODEL_DIR, "disease_model_float32.tflite")
tflite_utils.convert_to_tflite_float32(deployment_model, float32_path)


## 21. INT8 Quantization

Full-integer post-training quantization, calibrated on a representative sample of **training images only** (never validation/test/field-test — avoids calibration leakage).

In [ ]:
rep_dataset_fn = tflite_utils.build_representative_dataset_fn(
    config.TRAIN_DIR, config.IMAGE_SIZE,
    num_samples=config.REPRESENTATIVE_DATASET_SIZE,
    supported_extensions=config.SUPPORTED_EXTENSIONS,
    seed=config.SEED,
)
int8_path = os.path.join(config.FINAL_MODEL_DIR, "disease_model_int8.tflite")
tflite_utils.convert_to_tflite_int8(deployment_model, int8_path, rep_dataset_fn)


## 22. TFLite Validation

Inspects both TFLite models' actual input/output shape, dtype, and quantization parameters (never assumed), then compares Keras vs TFLite predictions on real test images, and benchmarks CPU inference latency.

In [ ]:
print("--- Float32 model ---")
tflite_utils.inspect_tflite_model(float32_path)
print("\n--- INT8 model ---")
tflite_utils.inspect_tflite_model(int8_path)


In [ ]:
import random as _random

sample_paths = []
for cls in sorted(os.listdir(config.TEST_DIR)):
    cls_dir = os.path.join(config.TEST_DIR, cls)
    if os.path.isdir(cls_dir):
        sample_paths.extend(os.path.join(cls_dir, f) for f in os.listdir(cls_dir))
_random.Random(config.SEED).shuffle(sample_paths)
sample_paths = sample_paths[:10]

print("=== Keras vs Float32 TFLite ===")
tflite_utils.compare_keras_vs_tflite(deployment_model, float32_path, sample_paths, class_names, config.IMAGE_SIZE)

print("\n=== Keras vs INT8 TFLite ===")
tflite_utils.compare_keras_vs_tflite(deployment_model, int8_path, sample_paths, class_names, config.IMAGE_SIZE)


In [ ]:
float32_latency = tflite_utils.benchmark_tflite_cpu(float32_path, config.IMAGE_SIZE)
int8_latency = tflite_utils.benchmark_tflite_cpu(int8_path, config.IMAGE_SIZE)
print(f"Float32 TFLite — mean: {float32_latency['mean_ms']:.2f} ms, median: {float32_latency['median_ms']:.2f} ms")
print(f"INT8 TFLite    — mean: {int8_latency['mean_ms']:.2f} ms, median: {int8_latency['median_ms']:.2f} ms")

float32_size = evaluation.get_model_file_size_mb(float32_path)
int8_size = evaluation.get_model_file_size_mb(int8_path)
print(f"\nFloat32 size: {float32_size:.2f} MB | INT8 size: {int8_size:.2f} MB")
if float32_size:
    print(f"Size reduction: {(1 - int8_size / float32_size) * 100:.1f}%")


## 23. Final Prediction Demo

Demonstrates `predict_disease()` (Keras) and `predict_tflite()` (INT8 TFLite) on a real test image, with image-quality check, confidence, and top-3 predictions — all real, actual model outputs.

In [ ]:
from src.prediction import predict_disease, print_prediction_result

demo_image_path = sample_paths[0] if sample_paths else None

if demo_image_path:
    print("Demo image:", demo_image_path)
    print("\n--- Keras prediction ---")
    keras_result = predict_disease(
        demo_image_path, deployment_model, class_names, config.IMAGE_SIZE,
        confidence_threshold=config.CONFIDENCE_THRESHOLD,
    )
    print_prediction_result(keras_result)

    print("\n--- INT8 TFLite prediction ---")
    tflite_result = tflite_utils.predict_tflite(
        demo_image_path, int8_path, class_names, config.IMAGE_SIZE,
        confidence_threshold=config.CONFIDENCE_THRESHOLD,
    )
    print(tflite_result)
else:
    print("No test images available for the demo.")


## 24. Final Deployment Package

Generates `class_names.txt`, `class_names.json`, `preprocessing_config.json`, and `model_metadata.json` into `Models/Final/` — completing the deployment package alongside the two `.tflite` files.

In [ ]:
class_names_txt = os.path.join(config.FINAL_MODEL_DIR, "class_names.txt")
with open(class_names_txt, "w") as f:
    f.write("\n".join(class_names))
print(f"Saved: {class_names_txt}")

class_names_json = {str(i): name for i, name in enumerate(class_names)}
utils.save_json(class_names_json, os.path.join(config.FINAL_MODEL_DIR, "class_names.json"))


In [ ]:
preprocessing_config = {
    "image_size": list(config.IMAGE_SIZE),
    "color_mode": "RGB",
    "normalization": (
        "Model-specific preprocessing is baked into the exported graph "
        "(Rescaling/Lambda layer inside the model). Supply raw [0,255] "
        "uint8 pixels at inference time."
    ),
    "model_type": DEPLOYMENT_MODEL_NAME,
    "float32_input_dtype": "float32",
    "float32_output_dtype": "float32",
    "int8_model_note": (
        "Read input/output dtype and quantization scale/zero_point directly "
        "from the INT8 .tflite file at inference time — do not assume uint8 vs int8."
    ),
    "confidence_threshold": config.CONFIDENCE_THRESHOLD,
    "class_names": class_names,
}
utils.save_json(preprocessing_config, os.path.join(config.FINAL_MODEL_DIR, "preprocessing_config.json"))


In [ ]:
float32_size = evaluation.get_model_file_size_mb(float32_path)
int8_size = evaluation.get_model_file_size_mb(int8_path)

metadata = {
    "model_name": DEPLOYMENT_MODEL_NAME,
    "training_dataset": "PlantVillage (subset) — see config.SELECTED_CLASSES",
    "num_classes": len(class_names),
    "class_names": class_names,
    "input_shape": list(config.IMAGE_SIZE) + [config.IMAGE_CHANNELS],
    "float32_model_size_mb": round(float32_size, 3) if float32_size else None,
    "int8_model_size_mb": round(int8_size, 3) if int8_size else None,
    "note": (
        "Test-set accuracy/precision/recall/F1 for this model are in "
        "Results/Metrics/model_comparison.csv and "
        f"Results/Metrics/{DEPLOYMENT_MODEL_NAME}_classification_report.txt. "
        "Not duplicated here to avoid drift."
    ),
}
utils.save_json(metadata, os.path.join(config.FINAL_MODEL_DIR, "model_metadata.json"))

print(f"\n[OK] Final deployment package ready in: {config.FINAL_MODEL_DIR}")
print(os.listdir(config.FINAL_MODEL_DIR))


---
## Acceptance checklist

Re-run this notebook top to bottom against your real dataset, then check off each item in the project spec's Final Acceptance Checklist (see the original project brief) against what actually printed above — every checkbox should map to a real, executed cell in this notebook, not an assumption.
